In [1]:
from google.cloud import pubsub_v1
import json
import os

# 1. Configuration from your .env
project_id = os.getenv("GCP_PROJECT_ID")
subscription_id = "bitcoin-price-sub" # The name from your Terraform

# 2. Initialize the Subscriber
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

print(f"🚀 Listening for Bitcoin data on {subscription_id}...")

def callback(message: pubsub_v1.subscriber.message.Message) -> None:
    """
    Process the incoming Bitcoin price message.
    """
    # Parse the data
    payload = json.loads(message.data.decode("utf-8"))
    
    # Senior View: Extract key metrics
    asset = payload['data']['asset']
    price_aud = payload['data']['price_aud']
    timestamp = payload['metadata']['ingested_at']
    
    print(f"✅ Received {asset.upper()}: ${price_aud:,.2f} AUD | Ingested: {timestamp}")
    
    # CRITICAL: Acknowledge the message so it's removed from the 'Unacked' graph
    message.ack()

# 3. Start the "Pull" flow
# We limit to 5 messages for this notebook test
streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)

try:
    # Keep the notebook cell alive for 30 seconds to catch messages
    streaming_pull_future.result(timeout=30)
except Exception as e:
    streaming_pull_future.cancel()
    print(f"\n⏹️ Stopped listening. (Note: {e})")

🚀 Listening for Bitcoin data on bitcoin-price-sub...
✅ Received BITCOIN: $97,257.00 AUD | Ingested: 2026-03-22T22:45:37.948315+00:00
✅ Received ETHEREUM: $2,936.50 AUD | Ingested: 2026-03-22T22:49:22.295974+00:00
✅ Received BITCOIN: $97,373.00 AUD | Ingested: 2026-03-22T22:50:23.568541+00:00
✅ Received ETHEREUM: $2,934.42 AUD | Ingested: 2026-03-22T22:47:20.227682+00:00
✅ Received ETHEREUM: $2,938.52 AUD | Ingested: 2026-03-22T22:50:23.597682+00:00
✅ Received BITCOIN: $97,336.00 AUD | Ingested: 2026-03-22T22:49:22.241981+00:00
✅ Received BITCOIN: $97,278.00 AUD | Ingested: 2026-03-22T22:47:20.171360+00:00
✅ Received ETHEREUM: $2,933.02 AUD | Ingested: 2026-03-22T22:45:38.015289+00:00


KeyboardInterrupt: 